# W05 — ETL ทำความสะอาดและลดความเสี่ยงข้อมูล

## จุดประสงค์การเรียนรู้
- ค้นหาและนับจำนวนแถวที่บันทึกซ้ำ อายุที่ผิดเกณฑ์ และเกรดเฉลี่ยที่ผิดเกณฑ์ พร้อมคำนวณร้อยละได้
- แก้ไขข้อมูลในตารางสำเนา โดยเก็บตารางข้อมูลต้นฉบับไว้โดยไม่แก้ไขได้
- ลบแถวที่บันทึกซ้ำ ปรับชื่อสาขาให้เขียนแบบเดียวกัน และจัดการค่าที่ผิดเกณฑ์อย่างมีเหตุผลได้
- แปลงวันที่ บันทึกผลก่อนและหลังการแก้ไข และเผยแพร่เฉพาะข้อมูลสรุปที่ไม่ระบุตัวบุคคลได้

## การเตรียมไฟล์ข้อมูล
- student_register_dirty.csv

อัปโหลดไฟล์ข้อมูลผ่านเมนู Files ในโปรแกรม Google Colab แล้วพิมพ์คำสั่งและรันผลลัพธ์ตามลำดับทีละขั้นตอน

## โครงสร้างชุดข้อมูล
- ไฟล์ student_register_dirty.csv: จำนวน 42 แถว 7 คอลัมน์ ได้แก่ student_id, name, major, age, gpa, phone, updated_date


## ส่วนที่ 2: ค้นหาและนับข้อผิดพลาด (ขั้นตอนที่ 1–11)

- เป้าหมายของส่วนนี้: นับให้ได้ว่าตารางมีข้อผิดพลาดแต่ละแบบกี่แถว โดยยังไม่แก้ไขข้อมูลใด
- เปรียบเทียบ: เหมือนแพทย์ที่ต้องตรวจร่างกายให้ครบและทราบอาการทั้งหมดก่อน จึงค่อยเริ่มรักษา
- สิ่งที่จะตรวจ: 1) แถวที่บันทึกซ้ำ 2) อายุที่ไม่อยู่ในช่วง 15–60 ปี 3) เกรดเฉลี่ยที่ไม่อยู่ในช่วง 0.00–4.00
- หลักการสำคัญ: ช่องที่ไม่ได้กรอกข้อมูล (ค่าว่าง) ไม่นับเป็นค่าผิดเกณฑ์ เพราะไม่ใช่การกรอกผิด แต่เป็นการไม่ได้กรอก จึงต้องแยกนับเป็นคนละปัญหา
- ผลที่ควรได้เมื่อจบส่วนนี้: แถวซ้ำ 2 แถว อายุผิดเกณฑ์ 1 แถว และเกรดเฉลี่ยผิดเกณฑ์ 1 แถว


## ขั้นตอนที่ 1: เรียกใช้เครื่องมือ pandas

pandas คือเครื่องมือของภาษา Python สำหรับทำงานกับตารางข้อมูล คล้ายโปรแกรม Excel ต้องเรียกใช้ก่อนทุกครั้งที่เริ่มทำงาน และตั้งชื่อย่อว่า pd เพื่อให้พิมพ์คำสั่งได้สั้นลง

```python
import pandas as pd
```

- `import pandas`: เรียกใช้เครื่องมือ pandas
- `as pd`: ตั้งชื่อย่อว่า pd ขั้นตอนต่อไปจะพิมพ์ pd แทนคำว่า pandas

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [1]:
import pandas as pd

### ผลลัพธ์และคำอธิบาย

```text
รันคำสั่งสำเร็จ (ไม่มีข้อความแจ้งข้อผิดพลาด)
```

ขั้นตอนนี้ไม่แสดงผลลัพธ์ หากไม่มีข้อความสีแดงปรากฏ แสดงว่าเรียกใช้สำเร็จ

**ข้อควรระวังในการเขียนคำสั่ง:** พิมพ์ด้วยตัวอักษรพิมพ์เล็กทั้งหมด และเมื่อ Colab เริ่มต้นใหม่ ต้องรันขั้นตอนนี้อีกครั้ง

**คำถามตรวจสอบความเข้าใจ:** ผู้เรียนทุกคนรันขั้นตอนนี้แล้วไม่มีข้อความแจ้งข้อผิดพลาด


## ขั้นตอนที่ 2: นำเข้าไฟล์ข้อมูลทะเบียนผู้เรียน

อ่านไฟล์ข้อมูลเข้ามาเก็บไว้ในตารางชื่อ df และกำหนดให้คอลัมน์หมายเลขโทรศัพท์เป็นข้อความ เพื่อไม่ให้เลข 0 ตัวแรกถูกตัดออก

```python
df = pd.read_csv('student_register_dirty.csv', dtype={'phone': 'string'})
df
```

- `df =`: ตั้งชื่อตารางที่อ่านเข้ามาว่า df ตารางนี้คือตารางข้อมูลต้นฉบับ ซึ่งจะไม่แก้ไขตลอดคาบ
- `pd.read_csv(...)`: อ่านไฟล์ CSV ตามชื่อที่ระบุ ชื่อไฟล์ต้องตรงกับไฟล์ที่อัปโหลดไว้ทุกตัวอักษร
- `dtype={'phone': 'string'}`: กำหนดชนิดข้อมูลของคอลัมน์ phone ให้เป็นข้อความ หากไม่กำหนด หมายเลข 0882903368 จะกลายเป็นตัวเลข 882903368
- `df`: บรรทัดสุดท้ายพิมพ์ชื่อตาราง เพื่อแสดงข้อมูลในตาราง

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [2]:
df = pd.read_csv('student_register_dirty.csv', dtype={'phone': 'string'})
df

,student_id,name,major,age,gpa,phone,updated_date
0,S001,สมชาย สุขสม,การตลาด,19.0,2.11,0882903368,2026-06-01
1,S002,อรุณ ศรีสุข,คอมพิวเตอร์ธุรกิจ,22.0,3.61,0845046288,2026-06-09
2,S003,วรรณา สุขสม,คอมพิวเตอร์ธุรกิจ,18.0,2.92,0877470852,2026-06-06
3,S004,ชัย สุขสม,การตลาด,18.0,NaN,0863778945,2026-06-04
4,S005,กมล ศรีสุข,คอมพิวเตอร์ธุรกิจ,19.0,3.34,0813749650,2026-06-05
5,S006,ศิริ ทองคำ,เทคโนโลยีสารสนเทศ,22.0,3.07,0856208603,2026-06-08
6,S007,อรุณ บุญมา,คอมพิวเตอร์ธุรกิจ,19.0,1.97,0823711300,2026-06-04
7,S008,ศิริ สุขสม,คอมพิวเตอร์ธุรกิจ,150.0,2.81,0891907998,2026-06-01
8,S009,ศิริ บุญมา,คอมพิวเตอร์ธุรกิจ,22.0,1.98,0898662305,2026-06-02
9,S010,ภูมิ แก้วมณี,การตลาด,18.0,2.80,0833960779,2026-06-07


### ผลลัพธ์และคำอธิบาย

| ลำดับแถว | student_id | name | major | age | gpa | phone | updated_date |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | S001 | สมชาย สุขสม | การตลาด | 19.0 | 2.11 | 0882903368 | 2026-06-01 |
| 1 | S002 | อรุณ ศรีสุข | คอมพิวเตอร์ธุรกิจ | 22.0 | 3.61 | 0845046288 | 2026-06-09 |
| 2 | S003 | วรรณา สุขสม | คอมพิวเตอร์ธุรกิจ | 18.0 | 2.92 | 0877470852 | 2026-06-06 |
| 3 | S004 | ชัย สุขสม | การตลาด | 18.0 | ว่าง | 0863778945 | 2026-06-04 |
| 4 | S005 | กมล ศรีสุข | คอมพิวเตอร์ธุรกิจ | 19.0 | 3.34 | 0813749650 | 2026-06-05 |
| 5 | S006 | ศิริ ทองคำ | เทคโนโลยีสารสนเทศ | 22.0 | 3.07 | 0856208603 | 2026-06-08 |
| 6 | S007 | อรุณ บุญมา | คอมพิวเตอร์ธุรกิจ | 19.0 | 1.97 | 0823711300 | 2026-06-04 |
| 7 | S008 | ศิริ สุขสม | คอมพิวเตอร์ธุรกิจ | 150.0 | 2.81 | 0891907998 | 2026-06-01 |
| 8 | S009 | ศิริ บุญมา | คอมพิวเตอร์ธุรกิจ | 22.0 | 1.98 | 0898662305 | 2026-06-02 |
| 9 | S010 | ภูมิ แก้วมณี | การตลาด | 18.0 | 2.8 | 0833960779 | 2026-06-07 |

แสดงตารางข้อมูล 42 แถว 7 คอลัมน์ ช่องที่แสดงคำว่า NaN หรือ <NA> คือช่องที่ไม่ได้กรอกข้อมูล (ค่าว่าง) และ 2 แถวสุดท้าย (S006 และ S021) เป็นแถวที่บันทึกซ้ำ

**ข้อควรระวังในการเขียนคำสั่ง:** หากพบข้อความ FileNotFoundError แสดงว่ายังไม่ได้อัปโหลดไฟล์ หรือพิมพ์ชื่อไฟล์ไม่ตรง ให้ตรวจสอบชื่อไฟล์ในแถบ Files ทางด้านซ้าย

**คำถามตรวจสอบความเข้าใจ:** ตารางมีกี่แถว และหมายเลขโทรศัพท์ยังมีเลข 0 นำหน้าหรือไม่ (คำตอบ: 42 แถว และยังมีเลข 0 นำหน้า)


## ขั้นตอนที่ 3: แสดงข้อมูลตัวอย่าง 5 แถวแรก

แสดงข้อมูลตัวอย่างเฉพาะ 5 แถวแรก เพื่อตรวจสอบว่าข้อมูลถูกอ่านเข้ามาครบทุกคอลัมน์ เหมาะสำหรับตารางที่มีข้อมูลจำนวนมาก

```python
df.head()
```

- `df`: ตารางที่ต้องการแสดงข้อมูล
- `.head()`: แสดงข้อมูลตัวอย่าง 5 แถวแรกของตาราง

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [3]:
df.head()

,student_id,name,major,age,gpa,phone,updated_date
0,S001,สมชาย สุขสม,การตลาด,19.0,2.11,0882903368,2026-06-01
1,S002,อรุณ ศรีสุข,คอมพิวเตอร์ธุรกิจ,22.0,3.61,0845046288,2026-06-09
2,S003,วรรณา สุขสม,คอมพิวเตอร์ธุรกิจ,18.0,2.92,0877470852,2026-06-06
3,S004,ชัย สุขสม,การตลาด,18.0,NaN,0863778945,2026-06-04
4,S005,กมล ศรีสุข,คอมพิวเตอร์ธุรกิจ,19.0,3.34,0813749650,2026-06-05


### ผลลัพธ์และคำอธิบาย

| ลำดับแถว | student_id | name | major | age | gpa | phone | updated_date |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | S001 | สมชาย สุขสม | การตลาด | 19.0 | 2.11 | 0882903368 | 2026-06-01 |
| 1 | S002 | อรุณ ศรีสุข | คอมพิวเตอร์ธุรกิจ | 22.0 | 3.61 | 0845046288 | 2026-06-09 |
| 2 | S003 | วรรณา สุขสม | คอมพิวเตอร์ธุรกิจ | 18.0 | 2.92 | 0877470852 | 2026-06-06 |
| 3 | S004 | ชัย สุขสม | การตลาด | 18.0 | ว่าง | 0863778945 | 2026-06-04 |
| 4 | S005 | กมล ศรีสุข | คอมพิวเตอร์ธุรกิจ | 19.0 | 3.34 | 0813749650 | 2026-06-05 |

แสดงผู้เรียนรหัส S001–S005 ช่อง gpa ของ S004 เป็น NaN หมายความว่าไม่ได้กรอกเกรดเฉลี่ย

**ข้อควรระวังในการเขียนคำสั่ง:** NaN หมายถึงไม่มีข้อมูล ไม่ได้หมายถึงเกรดเฉลี่ยเป็น 0 จึงห้ามตีความว่าผู้เรียนได้เกรดเฉลี่ย 0

**คำถามตรวจสอบความเข้าใจ:** ค่าว่าง (NaN) แตกต่างจากเกรดเฉลี่ย 0 อย่างไร


## ขั้นตอนที่ 4: นับจำนวนแถวทั้งหมด

นับว่าตารางมีกี่แถว แล้วเก็บผลไว้ในชื่อ row_count เพื่อใช้เป็นตัวหารเมื่อคำนวณร้อยละในขั้นตอนที่ 13

```python
row_count = len(df)
row_count
```

- `len(df)`: นับจำนวนแถวของตาราง df
- `row_count =`: เก็บจำนวนที่นับได้ไว้ในชื่อ row_count เพื่อเรียกใช้ภายหลัง
- `row_count`: บรรทัดสุดท้ายพิมพ์ชื่อ เพื่อแสดงค่าที่เก็บไว้

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [4]:
row_count = len(df)
row_count

42

### ผลลัพธ์และคำอธิบาย

```text
42
```

ได้ 42 แถว ซึ่งยังรวมแถวที่บันทึกซ้ำ 2 แถว

**ข้อควรระวังในการเขียนคำสั่ง:** 42 คือจำนวนแถว ไม่ใช่จำนวนผู้เรียน เพราะมีผู้เรียน 2 คนถูกบันทึกซ้ำ จำนวนผู้เรียนจริงคือ 40 คน

**คำถามตรวจสอบความเข้าใจ:** เหตุใดจำนวนแถวจึงเป็น 42 แต่จำนวนผู้เรียนจริงเป็น 40 คน


## ขั้นตอนที่ 5: นับแถวที่บันทึกซ้ำ

ตรวจหาแถวที่ข้อมูลเหมือนกับแถวก่อนหน้าทุกช่อง แล้วนับจำนวนแถวที่ซ้ำ เก็บไว้ในชื่อ duplicate_count

```python
duplicate_count = df.duplicated().sum()
duplicate_count
```

- `df.duplicated()`: ตรวจทีละแถวว่าซ้ำกับแถวที่อยู่ก่อนหน้าหรือไม่ ได้คำตอบเป็น True (ซ้ำ) หรือ False (ไม่ซ้ำ)
- `.sum()`: นับจำนวนคำตอบที่เป็น True คือจำนวนแถวที่ซ้ำ
- `duplicate_count =`: เก็บจำนวนแถวซ้ำไว้ใช้ในตารางรายงาน

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [5]:
duplicate_count = df.duplicated().sum()
duplicate_count

2

### ผลลัพธ์และคำอธิบาย

```text
2
```

ได้ 2 แถว คือแถวท้ายตารางของ S006 และ S021 ส่วนแถวแรกของผู้เรียนแต่ละคนไม่นับว่าซ้ำ

**ข้อควรระวังในการเขียนคำสั่ง:** ขั้นตอนนี้เป็นการนับเท่านั้น ยังไม่ได้ลบแถวซ้ำ การลบแถวซ้ำจะทำในขั้นตอนที่ 16

**คำถามตรวจสอบความเข้าใจ:** ผู้เรียน S006 ปรากฏในตาราง 2 แถว แต่เหตุใดจึงนับเป็นแถวซ้ำเพียง 1 แถว


## ขั้นตอนที่ 6: ตรวจว่าอายุอยู่ในช่วง 15–60 ปีหรือไม่

ตรวจอายุของผู้เรียนทีละคนว่าอยู่ระหว่าง 15 ถึง 60 ปีหรือไม่ แล้วนับว่ามีกี่แถวที่อยู่ในเกณฑ์และไม่อยู่ในเกณฑ์

```python
age_valid = df['age'].between(15, 60)
age_valid.value_counts()
```

- `df['age']`: เลือกเฉพาะคอลัมน์อายุ
- `.between(15, 60)`: ตรวจว่าอายุอยู่ระหว่าง 15 ถึง 60 ปีหรือไม่ (นับรวม 15 และ 60) ได้คำตอบ True = อยู่ในเกณฑ์ และ False = ไม่อยู่ในเกณฑ์
- `age_valid =`: เก็บคำตอบของทุกแถวไว้ในชื่อ age_valid
- `.value_counts()`: นับว่ามีคำตอบ True กี่แถว และ False กี่แถว

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [6]:
age_valid = df['age'].between(15, 60)
age_valid.value_counts()

age
True     40
False     2
Name: count, dtype: int64

### ผลลัพธ์และคำอธิบาย

```text
age
True     40
False     2
```

True 40 แถว และ False 2 แถว แถวที่เป็น False คือ S008 (อายุ 150 ปี) และ S011 (ไม่ได้กรอกอายุ)

**ข้อควรระวังในการเขียนคำสั่ง:** แถวที่ไม่ได้กรอกอายุจะได้คำตอบเป็น False ด้วย เพราะไม่มีตัวเลขให้ตรวจ จึงยังสรุปไม่ได้ว่าอายุผิดเกณฑ์ 2 แถว ต้องแยกค่าว่างออกในขั้นตอนถัดไป

**คำถามตรวจสอบความเข้าใจ:** คำตอบ False ทั้ง 2 แถวเกิดจากสาเหตุใดบ้าง


## ขั้นตอนที่ 7: แยกเฉพาะแถวที่อายุผิดเกณฑ์

เลือกเฉพาะแถวที่กรอกอายุไว้ แต่อายุไม่อยู่ในช่วง 15–60 ปี โดยไม่นับแถวที่ไม่ได้กรอกอายุ แล้วแสดงแถวนั้นออกมา

```python
age_bad = ~age_valid & df['age'].notna()
df[age_bad]
```

- `~age_valid`: กลับคำตอบจากขั้นตอนที่ 6 แถวที่ไม่อยู่ในเกณฑ์จะกลายเป็น True
- `df['age'].notna()`: ตรวจว่าช่องอายุมีการกรอกข้อมูลหรือไม่ (True = กรอกแล้ว)
- `&`: หมายถึง และ คือต้องเป็นจริงทั้ง 2 เงื่อนไข ได้แก่ ไม่อยู่ในเกณฑ์ และกรอกข้อมูลแล้ว
- `age_bad =`: เก็บผลไว้ในชื่อ age_bad (แถวที่อายุผิดเกณฑ์)
- `df[age_bad]`: แสดงเฉพาะแถวที่ age_bad เป็น True

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [7]:
age_bad = ~age_valid & df['age'].notna()
df[age_bad]

,student_id,name,major,age,gpa,phone,updated_date
7,S008,ศิริ สุขสม,คอมพิวเตอร์ธุรกิจ,150.0,2.81,0891907998,2026-06-01


### ผลลัพธ์และคำอธิบาย

| ลำดับแถว | student_id | name | major | age | gpa | phone | updated_date |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 7 | S008 | ศิริ สุขสม | คอมพิวเตอร์ธุรกิจ | 150.0 | 2.81 | 0891907998 | 2026-06-01 |

แสดงผู้เรียนเพียง 1 คน คือ S008 อายุ 150 ปี ส่วน S011 ที่ไม่ได้กรอกอายุไม่ปรากฏ เพราะไม่ใช่การกรอกผิด

**ข้อควรระวังในการเขียนคำสั่ง:** ใช้เครื่องหมาย & เท่านั้น ห้ามพิมพ์คำว่า and และต้องพิมพ์เครื่องหมาย ~ (อยู่บนแป้นเดียวกับเครื่องหมาย `) ไว้หน้า age_valid

**คำถามตรวจสอบความเข้าใจ:** เหตุใดผู้เรียน S011 ที่ไม่ได้กรอกอายุจึงไม่ปรากฏในผลลัพธ์


## ขั้นตอนที่ 8: นับจำนวนแถวที่อายุผิดเกณฑ์

นับจำนวนแถวที่อายุผิดเกณฑ์ แล้วเก็บไว้ในชื่อ age_count เพื่อนำไปใส่ในตารางรายงาน

```python
age_count = age_bad.sum()
age_count
```

- `age_bad.sum()`: นับจำนวน True ใน age_bad คือจำนวนแถวที่อายุผิดเกณฑ์
- `age_count =`: เก็บจำนวนที่นับได้ไว้ในชื่อ age_count

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [8]:
age_count = age_bad.sum()
age_count

1

### ผลลัพธ์และคำอธิบาย

```text
1
```

ได้ 1 แถว ตรงกับผลในขั้นตอนที่ 7 ที่แสดงผู้เรียน S008 เพียงคนเดียว

**ข้อควรระวังในการเขียนคำสั่ง:** ต้องนับจาก age_bad (ขั้นตอนที่ 7) ไม่ใช่ age_valid (ขั้นตอนที่ 6) หากได้ผลลัพธ์เป็น 40 แสดงว่านับจากชื่อที่ไม่ถูกต้อง

**คำถามตรวจสอบความเข้าใจ:** เหตุใดจึงได้ 1 แถว ไม่ใช่ 2 แถว


## ขั้นตอนที่ 9: ตรวจว่าเกรดเฉลี่ยอยู่ในช่วง 0–4 หรือไม่

ตรวจเกรดเฉลี่ยทีละคนว่าอยู่ระหว่าง 0.00 ถึง 4.00 หรือไม่ ใช้วิธีเดียวกับการตรวจอายุในขั้นตอนที่ 6

```python
gpa_valid = df['gpa'].between(0, 4)
gpa_valid.value_counts()
```

- `df['gpa']`: เลือกเฉพาะคอลัมน์เกรดเฉลี่ย
- `.between(0, 4)`: ตรวจว่าเกรดเฉลี่ยอยู่ระหว่าง 0 ถึง 4 หรือไม่ (นับรวม 0 และ 4)
- `gpa_valid =`: เก็บคำตอบของทุกแถวไว้ในชื่อ gpa_valid
- `.value_counts()`: นับจำนวนคำตอบ True และ False

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [9]:
gpa_valid = df['gpa'].between(0, 4)
gpa_valid.value_counts()

gpa
True     39
False     3
Name: count, dtype: int64

### ผลลัพธ์และคำอธิบาย

```text
gpa
True     39
False     3
```

True 39 แถว และ False 3 แถว ได้แก่ S019 (เกรดเฉลี่ย 4.75) และแถวที่ไม่ได้กรอกเกรดเฉลี่ยอีก 2 แถว (S004 และ S023)

**ข้อควรระวังในการเขียนคำสั่ง:** False 3 แถว ไม่ได้หมายถึงเกรดเฉลี่ยผิดเกณฑ์ 3 แถว เพราะรวมแถวที่ไม่ได้กรอกข้อมูลไว้ด้วย

**คำถามตรวจสอบความเข้าใจ:** คำตอบ False ทั้ง 3 แถวประกอบด้วยแถวใดบ้าง


## ขั้นตอนที่ 10: แยกเฉพาะแถวที่เกรดเฉลี่ยผิดเกณฑ์

เลือกเฉพาะแถวที่กรอกเกรดเฉลี่ยไว้ แต่ไม่อยู่ในช่วง 0–4 แล้วแสดงแถวนั้นออกมา ใช้วิธีเดียวกับขั้นตอนที่ 7

```python
gpa_bad = ~gpa_valid & df['gpa'].notna()
df[gpa_bad]
```

- `~gpa_valid`: กลับคำตอบ แถวที่ไม่อยู่ในเกณฑ์จะกลายเป็น True
- `df['gpa'].notna()`: ตรวจว่ากรอกเกรดเฉลี่ยแล้วหรือไม่
- `&`: ต้องเป็นจริงทั้ง 2 เงื่อนไข
- `gpa_bad =`: เก็บผลไว้ในชื่อ gpa_bad (แถวที่เกรดเฉลี่ยผิดเกณฑ์)
- `df[gpa_bad]`: แสดงเฉพาะแถวที่ gpa_bad เป็น True

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [10]:
gpa_bad = ~gpa_valid & df['gpa'].notna()
df[gpa_bad]

,student_id,name,major,age,gpa,phone,updated_date
18,S019,จิรา ศรีสุข,เทคโนโลยีสารสนเทศ,21.0,4.75,0878524460,2026-06-01


### ผลลัพธ์และคำอธิบาย

| ลำดับแถว | student_id | name | major | age | gpa | phone | updated_date |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 18 | S019 | จิรา ศรีสุข | เทคโนโลยีสารสนเทศ | 21.0 | 4.75 | 0878524460 | 2026-06-01 |

แสดงผู้เรียนเพียง 1 คน คือ S019 ที่มีเกรดเฉลี่ย 4.75

**ข้อควรระวังในการเขียนคำสั่ง:** ใช้เครื่องหมาย & เท่านั้น ห้ามพิมพ์คำว่า and เพราะจะเกิดข้อความแจ้งข้อผิดพลาด

**คำถามตรวจสอบความเข้าใจ:** เหตุใดแถวของ S004 ที่ไม่ได้กรอกเกรดเฉลี่ยจึงไม่ปรากฏในผลลัพธ์


## ขั้นตอนที่ 11: นับจำนวนแถวที่เกรดเฉลี่ยผิดเกณฑ์

นับจำนวนแถวที่เกรดเฉลี่ยผิดเกณฑ์ แล้วเก็บไว้ในชื่อ gpa_count เพื่อนำไปใส่ในตารางรายงาน

```python
gpa_count = gpa_bad.sum()
gpa_count
```

- `gpa_bad.sum()`: นับจำนวน True ใน gpa_bad
- `gpa_count =`: เก็บจำนวนที่นับได้ไว้ในชื่อ gpa_count

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [11]:
gpa_count = gpa_bad.sum()
gpa_count

1

### ผลลัพธ์และคำอธิบาย

```text
1
```

ได้ 1 แถว คือ S019 เมื่อจบขั้นตอนนี้จะมีตัวเลขครบ 3 จำนวนสำหรับรายงาน ได้แก่ แถวซ้ำ 2 อายุผิดเกณฑ์ 1 และเกรดเฉลี่ยผิดเกณฑ์ 1

**ข้อควรระวังในการเขียนคำสั่ง:** ต้องนับจาก gpa_bad ไม่ใช่ gpa_valid

**คำถามตรวจสอบความเข้าใจ:** ผู้เรียนได้ตัวเลขทั้ง 3 จำนวนตรงกับสไลด์หรือไม่ (2, 1 และ 1)


## ส่วนที่ 3: จัดทำรายงานปัญหา (ขั้นตอนที่ 12–14)

- เป้าหมายของส่วนนี้: นำตัวเลข 3 จำนวนที่นับได้มาจัดเป็นตารางรายงานที่อ่านเข้าใจง่าย และบันทึกเป็นไฟล์เพื่อส่งต่อได้
- ตารางรายงานมี 4 คอลัมน์: ชื่อปัญหา (issue) จำนวนที่พบ (count) จำนวนแถวทั้งหมด (denominator) และร้อยละ (percent)
- วิธีคิดร้อยละ: ร้อยละ = จำนวนที่พบ ÷ จำนวนแถวทั้งหมด × 100 เช่น อายุผิดเกณฑ์ 1 แถว จาก 42 แถว = 1 ÷ 42 × 100 ≈ 2.38%
- เหตุผลที่ใช้ 42 เป็นตัวหาร: ขณะตรวจยังไม่ได้ลบแถวซ้ำ ตารางข้อมูลต้นฉบับจึงยังมี 42 แถว รายงานต้องระบุว่าเป็นร้อยละของจำนวนแถว ไม่ใช่ร้อยละของจำนวนผู้เรียน


## ขั้นตอนที่ 12: สร้างตารางรายงานปัญหา

นำตัวเลข 3 จำนวนที่นับได้มาจัดเป็นตารางรายงาน 3 แถว แถวละ 1 ปัญหา

```python
issue_report = pd.DataFrame({
    'issue': ['แถวซ้ำ', 'อายุผิดเกณฑ์', 'GPA ผิดเกณฑ์'],
    'count': [duplicate_count, age_count, gpa_count],
    'denominator': [row_count, row_count, row_count]
})
issue_report
```

- `pd.DataFrame({ ... })`: สร้างตารางใหม่ โดยกำหนดชื่อคอลัมน์และข้อมูลของแต่ละคอลัมน์ไว้ภายในเครื่องหมาย { }
- `'issue': [...]`: คอลัมน์ชื่อปัญหา มี 3 รายการ
- `'count': [...]`: คอลัมน์จำนวนที่พบ ใช้ตัวเลขที่เก็บไว้จากขั้นตอนที่ 5, 8 และ 11
- `'denominator': [...]`: คอลัมน์ตัวหาร ใช้จำนวนแถวทั้งหมด (row_count = 42) ทั้ง 3 แถว
- `issue_report =`: ตั้งชื่อตารางรายงานว่า issue_report

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [12]:
issue_report = pd.DataFrame({
    'issue': ['แถวซ้ำ', 'อายุผิดเกณฑ์', 'GPA ผิดเกณฑ์'],
    'count': [duplicate_count, age_count, gpa_count],
    'denominator': [row_count, row_count, row_count]
})
issue_report

,issue,count,denominator
0,แถวซ้ำ,2,42
1,อายุผิดเกณฑ์,1,42
2,GPA ผิดเกณฑ์,1,42


### ผลลัพธ์และคำอธิบาย

| ลำดับแถว | issue | count | denominator |
| --- | --- | --- | --- |
| 0 | แถวซ้ำ | 2 | 42 |
| 1 | อายุผิดเกณฑ์ | 1 | 42 |
| 2 | GPA ผิดเกณฑ์ | 1 | 42 |

ได้ตาราง 3 แถว ได้แก่ แถวซ้ำ 2 อายุผิดเกณฑ์ 1 และ GPA ผิดเกณฑ์ 1 โดยตัวหารทุกแถวเป็น 42

**ข้อควรระวังในการเขียนคำสั่ง:** ลำดับรายการในทุกคอลัมน์ต้องตรงกัน รายการที่ 1 ของ issue ต้องคู่กับรายการที่ 1 ของ count และทุกคอลัมน์ต้องมีจำนวนรายการเท่ากัน (3 รายการ)

**คำถามตรวจสอบความเข้าใจ:** แถวอายุผิดเกณฑ์มีจำนวนที่พบเท่าใด และใช้ตัวหารเท่าใด


## ขั้นตอนที่ 13: เพิ่มคอลัมน์ร้อยละในตารางรายงาน

เพิ่มคอลัมน์ใหม่ชื่อ percent โดยคำนวณร้อยละของแต่ละแถวจาก จำนวนที่พบ ÷ ตัวหาร × 100

```python
issue_report['percent'] = issue_report['count'] / issue_report['denominator'] * 100
issue_report
```

- `issue_report['percent'] =`: สร้างคอลัมน์ใหม่ชื่อ percent ในตารางรายงาน
- `issue_report['count'] / issue_report['denominator']`: นำจำนวนที่พบหารด้วยตัวหาร ทีละแถว
- `* 100`: คูณด้วย 100 เพื่อให้เป็นร้อยละ

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [13]:
issue_report['percent'] = issue_report['count'] / issue_report['denominator'] * 100
issue_report

,issue,count,denominator,percent
0,แถวซ้ำ,2,42,4.761905
1,อายุผิดเกณฑ์,1,42,2.380952
2,GPA ผิดเกณฑ์,1,42,2.380952


### ผลลัพธ์และคำอธิบาย

| ลำดับแถว | issue | count | denominator | percent |
| --- | --- | --- | --- | --- |
| 0 | แถวซ้ำ | 2 | 42 | 4.761904761904762 |
| 1 | อายุผิดเกณฑ์ | 1 | 42 | 2.380952380952381 |
| 2 | GPA ผิดเกณฑ์ | 1 | 42 | 2.380952380952381 |

แถวซ้ำประมาณ 4.76% (2 ÷ 42 × 100) ส่วนอายุผิดเกณฑ์และเกรดเฉลี่ยผิดเกณฑ์อย่างละประมาณ 2.38% (1 ÷ 42 × 100)

**ข้อควรระวังในการเขียนคำสั่ง:** ไม่ควรนำร้อยละทั้ง 3 แถวมาบวกกัน เพราะผู้เรียน 1 คนอาจมีหลายปัญหาพร้อมกัน ผลรวมจึงไม่ใช่ร้อยละของผู้เรียนที่มีปัญหา

**คำถามตรวจสอบความเข้าใจ:** แสดงวิธีคำนวณร้อยละของแถวซ้ำ 2 แถว จากทั้งหมด 42 แถว


## ขั้นตอนที่ 14: บันทึกตารางรายงานเป็นไฟล์ CSV

บันทึกตารางรายงานเป็นไฟล์ชื่อ quality_issues.csv เพื่อดาวน์โหลดเก็บไว้หรือส่งต่อให้ผู้อื่น

```python
issue_report.to_csv('quality_issues.csv', index=False, encoding='utf-8-sig')
```

- `issue_report.to_csv('quality_issues.csv', ...)`: บันทึกตาราง issue_report เป็นไฟล์ CSV ตามชื่อที่กำหนด
- `index=False`: ไม่บันทึกเลขลำดับแถว 0, 1, 2 ลงในไฟล์ เพราะไม่มีความหมายในรายงาน
- `encoding='utf-8-sig'`: กำหนดรูปแบบการบันทึกให้ภาษาไทยแสดงผลถูกต้องเมื่อนำไฟล์ไปใช้ในโปรแกรม Excel

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [14]:
issue_report.to_csv('quality_issues.csv', index=False, encoding='utf-8-sig')

### ผลลัพธ์และคำอธิบาย

```text
รันคำสั่งสำเร็จ (ไม่มีข้อความแจ้งข้อผิดพลาด)
```

ขั้นตอนนี้ไม่แสดงตาราง ให้คลิกรูปโฟลเดอร์ทางด้านซ้าย จะพบไฟล์ quality_issues.csv (หากไม่พบ ให้คลิกปุ่มรีเฟรชในแถบ Files)

**ข้อควรระวังในการเขียนคำสั่ง:** ไฟล์ที่สร้างใน Colab จะถูกลบเมื่อปิดหน้าต่าง หากต้องการเก็บไว้ ให้คลิกขวาที่ไฟล์แล้วเลือก Download

**คำถามตรวจสอบความเข้าใจ:** พบไฟล์ quality_issues.csv ในแถบ Files ทางด้านซ้าย


## ส่วนที่ 4: แก้ไขข้อมูลในตารางสำเนา (ขั้นตอนที่ 15–22)

- เป้าหมายของส่วนนี้: แก้ไขข้อผิดพลาดที่พบในตารางสำเนาชื่อ clean และเก็บตารางข้อมูลต้นฉบับ df ไว้โดยไม่แก้ไข
- เหตุผลที่ต้องสร้างสำเนา: เหมือนการถ่ายเอกสารก่อนเขียนแก้ หากแก้ไขผิดยังมีต้นฉบับไว้ตรวจสอบ และเปรียบเทียบผลก่อนและหลังการแก้ไขได้
- สิ่งที่จะแก้ไข: 1) ลบแถวที่บันทึกซ้ำ 2) ปรับชื่อสาขาให้เขียนแบบเดียวกัน 3) ใส่คำว่า ไม่ระบุ ในช่องหมายเลขโทรศัพท์ที่ว่าง 4) เปลี่ยนอายุ 150 ปี ให้เป็นค่าว่าง
- เหตุผลที่ไม่ใส่ตัวเลขใหม่แทน: ยังไม่ทราบอายุจริงของผู้เรียน จึงไม่ควรกำหนดตัวเลขขึ้นเอง ให้เปลี่ยนเป็นค่าว่างไว้ก่อน แล้วตรวจสอบกับงานทะเบียนภายหลัง
- เหตุผลที่ต้องตรวจอายุอีกครั้ง: หลังลบแถวซ้ำ ตาราง clean เหลือ 40 แถว ผลการตรวจเดิมที่ทำจากตาราง df จำนวน 42 แถว จึงนำมาใช้กับตาราง clean ไม่ได้ ต้องตรวจใหม่จากตาราง clean


## ขั้นตอนที่ 15: สร้างตารางสำเนาสำหรับแก้ไข

สร้างตารางสำเนาชื่อ clean จากตาราง df การแก้ไขทั้งหมดต่อจากนี้จะทำในตาราง clean เท่านั้น ส่วนตาราง df ยังคงเป็นตารางข้อมูลต้นฉบับไว้เปรียบเทียบ

```python
clean = df.copy()
clean.head()
```

- `df.copy()`: สร้างสำเนาของตาราง df
- `clean =`: ตั้งชื่อตารางสำเนาว่า clean
- `clean.head()`: แสดงข้อมูลตัวอย่าง 5 แถวแรกของตารางสำเนา

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [15]:
clean = df.copy()
clean.head()

,student_id,name,major,age,gpa,phone,updated_date
0,S001,สมชาย สุขสม,การตลาด,19.0,2.11,0882903368,2026-06-01
1,S002,อรุณ ศรีสุข,คอมพิวเตอร์ธุรกิจ,22.0,3.61,0845046288,2026-06-09
2,S003,วรรณา สุขสม,คอมพิวเตอร์ธุรกิจ,18.0,2.92,0877470852,2026-06-06
3,S004,ชัย สุขสม,การตลาด,18.0,NaN,0863778945,2026-06-04
4,S005,กมล ศรีสุข,คอมพิวเตอร์ธุรกิจ,19.0,3.34,0813749650,2026-06-05


### ผลลัพธ์และคำอธิบาย

| ลำดับแถว | student_id | name | major | age | gpa | phone | updated_date |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | S001 | สมชาย สุขสม | การตลาด | 19.0 | 2.11 | 0882903368 | 2026-06-01 |
| 1 | S002 | อรุณ ศรีสุข | คอมพิวเตอร์ธุรกิจ | 22.0 | 3.61 | 0845046288 | 2026-06-09 |
| 2 | S003 | วรรณา สุขสม | คอมพิวเตอร์ธุรกิจ | 18.0 | 2.92 | 0877470852 | 2026-06-06 |
| 3 | S004 | ชัย สุขสม | การตลาด | 18.0 | ว่าง | 0863778945 | 2026-06-04 |
| 4 | S005 | กมล ศรีสุข | คอมพิวเตอร์ธุรกิจ | 19.0 | 3.34 | 0813749650 | 2026-06-05 |

5 แถวแรกเหมือนกับตาราง df ทุกประการ เพราะยังไม่ได้แก้ไขข้อมูลใด

**ข้อควรระวังในการเขียนคำสั่ง:** ต้องพิมพ์ .copy() หากพิมพ์เพียง clean = df การแก้ไขตาราง clean จะส่งผลให้ตาราง df ถูกแก้ไขไปด้วย

**คำถามตรวจสอบความเข้าใจ:** ตาราง df และตาราง clean ทำหน้าที่ต่างกันอย่างไร


## ขั้นตอนที่ 16: ลบแถวที่บันทึกซ้ำ

ลบแถวที่ข้อมูลซ้ำกับแถวก่อนหน้าทุกช่องออกจากตาราง clean โดยเก็บแถวแรกไว้

```python
clean = clean.drop_duplicates()
clean
```

- `clean.drop_duplicates()`: ลบแถวที่ซ้ำ เหลือไว้เพียงแถวแรกของข้อมูลแต่ละชุด
- `clean =`: บันทึกผลกลับไปที่ตาราง clean หากไม่เขียนส่วนนี้ ตารางจะไม่เปลี่ยนแปลง
- `clean`: บรรทัดสุดท้ายแสดงตาราง clean หลังลบแถวซ้ำ

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [16]:
clean = clean.drop_duplicates()
clean

,student_id,name,major,age,gpa,phone,updated_date
0,S001,สมชาย สุขสม,การตลาด,19.0,2.11,0882903368,2026-06-01
1,S002,อรุณ ศรีสุข,คอมพิวเตอร์ธุรกิจ,22.0,3.61,0845046288,2026-06-09
2,S003,วรรณา สุขสม,คอมพิวเตอร์ธุรกิจ,18.0,2.92,0877470852,2026-06-06
3,S004,ชัย สุขสม,การตลาด,18.0,NaN,0863778945,2026-06-04
4,S005,กมล ศรีสุข,คอมพิวเตอร์ธุรกิจ,19.0,3.34,0813749650,2026-06-05
5,S006,ศิริ ทองคำ,เทคโนโลยีสารสนเทศ,22.0,3.07,0856208603,2026-06-08
6,S007,อรุณ บุญมา,คอมพิวเตอร์ธุรกิจ,19.0,1.97,0823711300,2026-06-04
7,S008,ศิริ สุขสม,คอมพิวเตอร์ธุรกิจ,150.0,2.81,0891907998,2026-06-01
8,S009,ศิริ บุญมา,คอมพิวเตอร์ธุรกิจ,22.0,1.98,0898662305,2026-06-02
9,S010,ภูมิ แก้วมณี,การตลาด,18.0,2.80,0833960779,2026-06-07


### ผลลัพธ์และคำอธิบาย

| ลำดับแถว | student_id | name | major | age | gpa | phone | updated_date |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | S001 | สมชาย สุขสม | การตลาด | 19.0 | 2.11 | 0882903368 | 2026-06-01 |
| 1 | S002 | อรุณ ศรีสุข | คอมพิวเตอร์ธุรกิจ | 22.0 | 3.61 | 0845046288 | 2026-06-09 |
| 2 | S003 | วรรณา สุขสม | คอมพิวเตอร์ธุรกิจ | 18.0 | 2.92 | 0877470852 | 2026-06-06 |
| 3 | S004 | ชัย สุขสม | การตลาด | 18.0 | ว่าง | 0863778945 | 2026-06-04 |
| 4 | S005 | กมล ศรีสุข | คอมพิวเตอร์ธุรกิจ | 19.0 | 3.34 | 0813749650 | 2026-06-05 |
| 5 | S006 | ศิริ ทองคำ | เทคโนโลยีสารสนเทศ | 22.0 | 3.07 | 0856208603 | 2026-06-08 |
| 6 | S007 | อรุณ บุญมา | คอมพิวเตอร์ธุรกิจ | 19.0 | 1.97 | 0823711300 | 2026-06-04 |
| 7 | S008 | ศิริ สุขสม | คอมพิวเตอร์ธุรกิจ | 150.0 | 2.81 | 0891907998 | 2026-06-01 |
| 8 | S009 | ศิริ บุญมา | คอมพิวเตอร์ธุรกิจ | 22.0 | 1.98 | 0898662305 | 2026-06-02 |
| 9 | S010 | ภูมิ แก้วมณี | การตลาด | 18.0 | 2.8 | 0833960779 | 2026-06-07 |

ตาราง clean เหลือ 40 แถว (ตรวจสอบได้จากข้อความใต้ตาราง) แถวซ้ำของ S006 และ S021 ถูกลบออก ส่วนตาราง df ยังมี 42 แถวเหมือนเดิม

**ข้อควรระวังในการเขียนคำสั่ง:** ต้องเขียน clean = นำหน้า เพื่อบันทึกผลการลบ หากรันเพียง clean.drop_duplicates() ตาราง clean จะยังมี 42 แถว

**คำถามตรวจสอบความเข้าใจ:** ตาราง clean และตาราง df มีจำนวนแถวเท่าใด เพราะเหตุใด


## ขั้นตอนที่ 17: ลบอักขระช่องว่างส่วนเกินหน้าและท้ายชื่อสาขา

ลบอักขระช่องว่างส่วนเกินที่อยู่หน้าและท้ายชื่อสาขา เพราะคอมพิวเตอร์ถือว่า 'การตลาด' กับ 'การตลาด ' (มีอักขระช่องว่างต่อท้ายข้อความ) เป็นคนละคำ

```python
clean['major'] = clean['major'].str.strip()
clean['major'].value_counts()
```

- `clean['major']`: เลือกคอลัมน์สาขาวิชา
- `.str.strip()`: ลบอักขระช่องว่างหน้าและท้ายข้อความ แต่ไม่ลบช่องว่างที่อยู่ตรงกลาง
- `clean['major'] =`: บันทึกผลกลับไปที่คอลัมน์เดิม
- `.value_counts()`: นับจำนวนผู้เรียนของแต่ละชื่อสาขา

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [17]:
clean['major'] = clean['major'].str.strip()
clean['major'].value_counts()

major
คอมพิวเตอร์ธุรกิจ     14
เทคโนโลยีสารสนเทศ     14
การตลาด               10
IT                     1
เทคโนโลยี สารสนเทศ     1
Name: count, dtype: int64

### ผลลัพธ์และคำอธิบาย

```text
major
คอมพิวเตอร์ธุรกิจ     14
เทคโนโลยีสารสนเทศ     14
การตลาด               10
IT                     1
เทคโนโลยี สารสนเทศ     1
```

ยังพบชื่อสาขา 5 แบบเหมือนเดิม เพราะชุดข้อมูลนี้ไม่มีอักขระช่องว่างส่วนเกินหน้าและท้ายข้อความ และยังพบ IT (1 คน) กับ เทคโนโลยี สารสนเทศ (1 คน) ซึ่งเป็นสาขาเดียวกับ เทคโนโลยีสารสนเทศ

**ข้อควรระวังในการเขียนคำสั่ง:** ควรทำขั้นตอนนี้ทุกครั้งแม้ผลจะไม่เปลี่ยน เพราะข้อมูลจริงมักมีอักขระช่องว่างต่อท้ายข้อความ ส่วนช่องว่างตรงกลางคำต้องแก้ไขในขั้นตอนถัดไป

**คำถามตรวจสอบความเข้าใจ:** ชื่อสาขาใดบ้างที่หมายถึงสาขาเดียวกัน


## ขั้นตอนที่ 18: ปรับชื่อสาขาให้เขียนแบบเดียวกัน

เปลี่ยนชื่อสาขาที่เขียนต่างกันให้เป็นชื่อเดียวกัน คือ เทคโนโลยีสารสนเทศ เพื่อให้นับจำนวนผู้เรียนแต่ละสาขาได้ถูกต้อง

```python
clean['major'] = clean['major'].replace({'IT': 'เทคโนโลยีสารสนเทศ', 'เทคโนโลยี สารสนเทศ': 'เทคโนโลยีสารสนเทศ'})
clean['major'].value_counts()
```

- `.replace({...})`: แทนที่ค่าเดิมด้วยค่าใหม่ ตามคู่ที่กำหนดไว้ภายในเครื่องหมาย { }
- `'IT': 'เทคโนโลยีสารสนเทศ'`: ด้านซ้ายของเครื่องหมาย : คือค่าเดิม ด้านขวาคือค่าใหม่ หมายถึงเปลี่ยน IT เป็น เทคโนโลยีสารสนเทศ
- `'เทคโนโลยี สารสนเทศ': 'เทคโนโลยีสารสนเทศ'`: เปลี่ยนชื่อที่มีช่องว่างตรงกลางให้เป็นชื่อที่ถูกต้อง
- `.value_counts()`: นับจำนวนผู้เรียนแต่ละสาขาอีกครั้งเพื่อตรวจผล

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [18]:
clean['major'] = clean['major'].replace({'IT': 'เทคโนโลยีสารสนเทศ', 'เทคโนโลยี สารสนเทศ': 'เทคโนโลยีสารสนเทศ'})
clean['major'].value_counts()

major
เทคโนโลยีสารสนเทศ    16
คอมพิวเตอร์ธุรกิจ    14
การตลาด              10
Name: count, dtype: int64

### ผลลัพธ์และคำอธิบาย

```text
major
เทคโนโลยีสารสนเทศ    16
คอมพิวเตอร์ธุรกิจ    14
การตลาด              10
```

เหลือชื่อสาขา 3 แบบ ได้แก่ เทคโนโลยีสารสนเทศ 16 คน คอมพิวเตอร์ธุรกิจ 14 คน และการตลาด 10 คน รวม 40 คน

**ข้อควรระวังในการเขียนคำสั่ง:** ต้องพิมพ์ค่าเดิมให้ตรงกับข้อมูลทุกตัวอักษร รวมทั้งช่องว่างตรงกลาง หากพิมพ์ไม่ตรง ชื่อนั้นจะไม่ถูกเปลี่ยน

**คำถามตรวจสอบความเข้าใจ:** หลังแก้ไข สาขาเทคโนโลยีสารสนเทศมีผู้เรียนกี่คน (คำตอบ: 16 คน)


## ขั้นตอนที่ 19: ระบุคำว่า ไม่ระบุ ในช่องหมายเลขโทรศัพท์ที่ว่าง

ใส่คำว่า ไม่ระบุ ในช่องหมายเลขโทรศัพท์ที่ว่าง เพื่อให้ผู้ใช้ข้อมูลทราบว่าผู้เรียนคนนี้ไม่ได้ให้หมายเลขโทรศัพท์ไว้

```python
clean['phone'] = clean['phone'].fillna('ไม่ระบุ')
clean[clean['phone'] == 'ไม่ระบุ']
```

- `.fillna('ไม่ระบุ')`: ใส่ข้อความ ไม่ระบุ เฉพาะช่องที่ว่าง ช่องที่มีหมายเลขโทรศัพท์อยู่แล้วจะไม่เปลี่ยน
- `clean['phone'] =`: บันทึกผลกลับไปที่คอลัมน์ phone
- `clean[clean['phone'] == 'ไม่ระบุ']`: แสดงเฉพาะแถวที่หมายเลขโทรศัพท์เป็นคำว่า ไม่ระบุ เพื่อตรวจผล

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [19]:
clean['phone'] = clean['phone'].fillna('ไม่ระบุ')
clean[clean['phone'] == 'ไม่ระบุ']

,student_id,name,major,age,gpa,phone,updated_date
15,S016,จิรา สุขสม,เทคโนโลยีสารสนเทศ,17.0,2.33,ไม่ระบุ,2026-06-09


### ผลลัพธ์และคำอธิบาย

| ลำดับแถว | student_id | name | major | age | gpa | phone | updated_date |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 15 | S016 | จิรา สุขสม | เทคโนโลยีสารสนเทศ | 17.0 | 2.33 | ไม่ระบุ | 2026-06-09 |

แสดงผู้เรียน 1 คน คือ S016 ซึ่งช่องหมายเลขโทรศัพท์เปลี่ยนจากค่าว่างเป็น ไม่ระบุ

**ข้อควรระวังในการเขียนคำสั่ง:** ห้ามใส่หมายเลขโทรศัพท์ขึ้นเอง เช่น 0000000000 เพราะจะทำให้ผู้อื่นเข้าใจผิดว่าเป็นหมายเลขจริง และต้องใช้เครื่องหมาย == (เท่ากับ 2 ตัว) ในบรรทัดที่ 2

**คำถามตรวจสอบความเข้าใจ:** เหตุใดจึงใส่คำว่า ไม่ระบุ แทนการใส่หมายเลขโทรศัพท์


## ขั้นตอนที่ 20: ตรวจอายุในตารางสำเนาอีกครั้ง

ตรวจอายุอีกครั้ง แต่ครั้งนี้ตรวจจากตาราง clean เพราะจะแก้ไขในตาราง clean ซึ่งขณะนี้เหลือ 40 แถว

```python
age_valid = clean['age'].between(15, 60)
age_valid.value_counts()
```

- `clean['age']`: เลือกคอลัมน์อายุจากตาราง clean (ไม่ใช่ตาราง df)
- `.between(15, 60)`: ตรวจว่าอายุอยู่ระหว่าง 15 ถึง 60 ปีหรือไม่
- `age_valid =`: เก็บคำตอบใหม่แทนที่ age_valid เดิม
- `.value_counts()`: นับจำนวนคำตอบ True และ False

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [20]:
age_valid = clean['age'].between(15, 60)
age_valid.value_counts()

age
True     38
False     2
Name: count, dtype: int64

### ผลลัพธ์และคำอธิบาย

```text
age
True     38
False     2
```

True 38 แถว และ False 2 แถว (S008 อายุ 150 ปี และ S011 ไม่ได้กรอกอายุ) รวมเป็น 40 แถว เท่ากับจำนวนแถวของตาราง clean

**ข้อควรระวังในการเขียนคำสั่ง:** ต้องตรวจจาก clean['age'] หากใช้ df['age'] จะได้ผล 42 แถว ซึ่งไม่ตรงกับตาราง clean และจะเกิดข้อความแจ้งข้อผิดพลาดในขั้นตอนที่ 22

**คำถามตรวจสอบความเข้าใจ:** เหตุใดต้องตรวจอายุใหม่ ทั้งที่ตรวจไปแล้วในขั้นตอนที่ 6


## ขั้นตอนที่ 21: แยกแถวที่อายุผิดเกณฑ์ในตารางสำเนา

เลือกเฉพาะแถวในตาราง clean ที่กรอกอายุไว้แต่ผิดเกณฑ์ แล้วแสดงแถวนั้นออกมา ใช้วิธีเดียวกับขั้นตอนที่ 7

```python
age_bad = ~age_valid & clean['age'].notna()
clean[age_bad]
```

- `~age_valid`: แถวที่ไม่อยู่ในเกณฑ์
- `clean['age'].notna()`: แถวที่กรอกอายุแล้ว
- `&`: ต้องเป็นจริงทั้ง 2 เงื่อนไข
- `age_bad =`: เก็บผลไว้ในชื่อ age_bad
- `clean[age_bad]`: แสดงเฉพาะแถวที่อายุผิดเกณฑ์

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [21]:
age_bad = ~age_valid & clean['age'].notna()
clean[age_bad]

,student_id,name,major,age,gpa,phone,updated_date
7,S008,ศิริ สุขสม,คอมพิวเตอร์ธุรกิจ,150.0,2.81,0891907998,2026-06-01


### ผลลัพธ์และคำอธิบาย

| ลำดับแถว | student_id | name | major | age | gpa | phone | updated_date |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 7 | S008 | ศิริ สุขสม | คอมพิวเตอร์ธุรกิจ | 150.0 | 2.81 | 0891907998 | 2026-06-01 |

แสดงผู้เรียน 1 คน คือ S008 อายุ 150 ปี ซึ่งเป็นแถวที่จะแก้ไขในขั้นตอนถัดไป

**ข้อควรระวังในการเขียนคำสั่ง:** ตรวจให้แน่ใจว่าผลลัพธ์มีเพียง S008 ก่อนแก้ไข เพื่อป้องกันการแก้ไขแถวที่ถูกต้องอยู่แล้ว

**คำถามตรวจสอบความเข้าใจ:** แถวที่จะแก้ไขในขั้นตอนถัดไปคือผู้เรียนรหัสใด


## ขั้นตอนที่ 22: เปลี่ยนอายุที่ผิดเกณฑ์ให้เป็นค่าว่าง

เปลี่ยนอายุ 150 ปี ให้เป็นค่าว่าง เพราะยังไม่ทราบอายุจริง จึงพักไว้ก่อนเพื่อรอตรวจสอบกับงานทะเบียน

```python
clean.loc[age_bad, 'age'] = pd.NA
clean[age_bad]
```

- `clean.loc[age_bad, 'age']`: ระบุตำแหน่งที่จะแก้ไข คือแถวที่ age_bad เป็น True และคอลัมน์ age
- `= pd.NA`: เปลี่ยนค่าในตำแหน่งนั้นให้เป็นค่าว่าง
- `clean[age_bad]`: แสดงแถวเดิมอีกครั้งเพื่อตรวจว่าแก้ไขแล้ว

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [22]:
clean.loc[age_bad, 'age'] = pd.NA
clean[age_bad]

,student_id,name,major,age,gpa,phone,updated_date
7,S008,ศิริ สุขสม,คอมพิวเตอร์ธุรกิจ,NaN,2.81,0891907998,2026-06-01


### ผลลัพธ์และคำอธิบาย

| ลำดับแถว | student_id | name | major | age | gpa | phone | updated_date |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 7 | S008 | ศิริ สุขสม | คอมพิวเตอร์ธุรกิจ | ว่าง | 2.81 | 0891907998 | 2026-06-01 |

แถวของ S008 ช่องอายุเปลี่ยนจาก 150 เป็น NaN (ค่าว่าง) ส่วนข้อมูลช่องอื่นของผู้เรียนคนนี้ไม่เปลี่ยนแปลง

**ข้อควรระวังในการเขียนคำสั่ง:** ไม่ใส่ค่าเฉลี่ยหรือตัวเลขอื่นแทน เพราะยังไม่มีหลักฐานว่าอายุจริงคือเท่าใด การกำหนดตัวเลขขึ้นเองจะทำให้ข้อมูลคลาดเคลื่อนจากความเป็นจริง

**คำถามตรวจสอบความเข้าใจ:** เหตุใดจึงเปลี่ยนเป็นค่าว่าง แทนการใส่อายุที่คาดว่าน่าจะถูกต้อง


---

# คาบเรียนที่ 2 (ขั้นตอนที่ 23–31)


## ส่วนที่ 2: แก้ไขเกรดเฉลี่ยที่ผิดเกณฑ์ (ขั้นตอนที่ 23–25)

- เป้าหมายของส่วนนี้: แก้ไขเกรดเฉลี่ย 4.75 ของผู้เรียน S019 ในตาราง clean ด้วยวิธีเดียวกับการแก้ไขอายุในวันจันทร์
- ทบทวนวิธีจากวันจันทร์: ตรวจช่วงค่า (ขั้นตอนที่ 20) → แยกแถวที่ผิดเกณฑ์ (ขั้นตอนที่ 21) → เปลี่ยนเป็นค่าว่าง (ขั้นตอนที่ 22) วันนี้ใช้ 3 ขั้นตอนเดิม แต่เปลี่ยนจากคอลัมน์ age เป็น gpa และเปลี่ยนช่วงเป็น 0–4
- เหตุผลที่ไม่ปรับเป็น 4.00: เกรดเฉลี่ย 4.75 อาจเกิดจากการพิมพ์ผิดจาก 3.75 หรือ 2.75 ก็ได้ จึงไม่ควรเลือกตัวเลขแทนเอง ให้เปลี่ยนเป็นค่าว่างแล้วตรวจสอบกับงานทะเบียน
- ผลที่ควรได้เมื่อจบส่วนนี้: S019 มีเกรดเฉลี่ยเป็นค่าว่าง และเกรดเฉลี่ยที่ว่างในตาราง clean รวมเป็น 3 แถว


## ขั้นตอนที่ 23: ตรวจเกรดเฉลี่ยในตารางสำเนาอีกครั้ง

ตรวจเกรดเฉลี่ยจากตาราง clean ว่าอยู่ในช่วง 0–4 หรือไม่ ใช้วิธีเดียวกับการตรวจอายุในขั้นตอนที่ 20

```python
gpa_valid = clean['gpa'].between(0, 4)
gpa_valid.value_counts()
```

- `clean['gpa']`: เลือกคอลัมน์เกรดเฉลี่ยจากตาราง clean (ไม่ใช่ตาราง df)
- `.between(0, 4)`: ตรวจว่าเกรดเฉลี่ยอยู่ระหว่าง 0 ถึง 4 หรือไม่ (นับรวม 0 และ 4)
- `gpa_valid =`: เก็บคำตอบใหม่แทนที่ gpa_valid เดิมจากวันจันทร์
- `.value_counts()`: นับจำนวนคำตอบ True และ False

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [23]:
gpa_valid = clean['gpa'].between(0, 4)
gpa_valid.value_counts()

gpa
True     37
False     3
Name: count, dtype: int64

### ผลลัพธ์และคำอธิบาย

```text
gpa
True     37
False     3
```

True 37 แถว และ False 3 แถว ได้แก่ S019 (เกรดเฉลี่ย 4.75) และแถวที่ไม่ได้กรอกเกรดเฉลี่ยอีก 2 แถว (S004 และ S023) รวมเป็น 40 แถว เท่ากับจำนวนแถวของตาราง clean

**ข้อควรระวังในการเขียนคำสั่ง:** ต้องตรวจจาก clean['gpa'] หากใช้ df['gpa'] จะได้ผล 42 แถว ซึ่งไม่ตรงกับตาราง clean และจะเกิดข้อความแจ้งข้อผิดพลาดในขั้นตอนที่ 25

**คำถามตรวจสอบความเข้าใจ:** คำตอบ False ทั้ง 3 แถวประกอบด้วยแถวใดบ้าง


## ขั้นตอนที่ 24: แยกแถวที่เกรดเฉลี่ยผิดเกณฑ์ในตารางสำเนา

เลือกเฉพาะแถวในตาราง clean ที่กรอกเกรดเฉลี่ยไว้แต่ผิดเกณฑ์ แล้วแสดงแถวนั้นออกมา ใช้วิธีเดียวกับขั้นตอนที่ 21

```python
gpa_bad = ~gpa_valid & clean['gpa'].notna()
clean[gpa_bad]
```

- `~gpa_valid`: แถวที่ไม่อยู่ในเกณฑ์
- `clean['gpa'].notna()`: แถวที่กรอกเกรดเฉลี่ยแล้ว
- `&`: ต้องเป็นจริงทั้ง 2 เงื่อนไข
- `gpa_bad =`: เก็บผลไว้ในชื่อ gpa_bad
- `clean[gpa_bad]`: แสดงเฉพาะแถวที่เกรดเฉลี่ยผิดเกณฑ์

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [24]:
gpa_bad = ~gpa_valid & clean['gpa'].notna()
clean[gpa_bad]

,student_id,name,major,age,gpa,phone,updated_date
18,S019,จิรา ศรีสุข,เทคโนโลยีสารสนเทศ,21.0,4.75,0878524460,2026-06-01


### ผลลัพธ์และคำอธิบาย

| ลำดับแถว | student_id | name | major | age | gpa | phone | updated_date |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 18 | S019 | จิรา ศรีสุข | เทคโนโลยีสารสนเทศ | 21.0 | 4.75 | 0878524460 | 2026-06-01 |

แสดงผู้เรียน 1 คน คือ S019 เกรดเฉลี่ย 4.75 ซึ่งเป็นแถวที่จะแก้ไขในขั้นตอนถัดไป

**ข้อควรระวังในการเขียนคำสั่ง:** ตรวจให้แน่ใจว่าผลลัพธ์มีเพียง S019 ก่อนแก้ไข เพื่อป้องกันการแก้ไขแถวที่ถูกต้องอยู่แล้ว

**คำถามตรวจสอบความเข้าใจ:** เหตุใดแถวของ S004 และ S023 ที่ไม่ได้กรอกเกรดเฉลี่ยจึงไม่ปรากฏในผลลัพธ์


## ขั้นตอนที่ 25: เปลี่ยนเกรดเฉลี่ยที่ผิดเกณฑ์ให้เป็นค่าว่าง

เปลี่ยนเกรดเฉลี่ย 4.75 ให้เป็นค่าว่าง เพราะยังไม่ทราบเกรดเฉลี่ยจริง จึงพักไว้ก่อนเพื่อรอตรวจสอบกับงานทะเบียน

```python
clean.loc[gpa_bad, 'gpa'] = pd.NA
clean[gpa_bad]
```

- `clean.loc[gpa_bad, 'gpa']`: ระบุตำแหน่งที่จะแก้ไข คือแถวที่ gpa_bad เป็น True และคอลัมน์ gpa
- `= pd.NA`: เปลี่ยนค่าในตำแหน่งนั้นให้เป็นค่าว่าง
- `clean[gpa_bad]`: แสดงแถวเดิมอีกครั้งเพื่อตรวจว่าแก้ไขแล้ว

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [25]:
clean.loc[gpa_bad, 'gpa'] = pd.NA
clean[gpa_bad]

,student_id,name,major,age,gpa,phone,updated_date
18,S019,จิรา ศรีสุข,เทคโนโลยีสารสนเทศ,21.0,NaN,0878524460,2026-06-01


### ผลลัพธ์และคำอธิบาย

| ลำดับแถว | student_id | name | major | age | gpa | phone | updated_date |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 18 | S019 | จิรา ศรีสุข | เทคโนโลยีสารสนเทศ | 21.0 | ว่าง | 0878524460 | 2026-06-01 |

แถวของ S019 ช่องเกรดเฉลี่ยเปลี่ยนจาก 4.75 เป็น NaN (ค่าว่าง) ส่วนข้อมูลช่องอื่นไม่เปลี่ยนแปลง

**ข้อควรระวังในการเขียนคำสั่ง:** ไม่ปรับเป็น 4.00 หรือใส่ค่าเฉลี่ยแทน เพราะยังไม่มีหลักฐานว่าเกรดเฉลี่ยจริงคือเท่าใด การกำหนดตัวเลขขึ้นเองจะทำให้ข้อมูลคลาดเคลื่อนจากความเป็นจริง

**คำถามตรวจสอบความเข้าใจ:** หลังแก้ไข ตาราง clean มีเกรดเฉลี่ยที่ว่างรวมกี่แถว (คำตอบ: 3 แถว)


## ส่วนที่ 3: แปลงวันที่และบันทึกการแก้ไข (ขั้นตอนที่ 26–28)

- เป้าหมายของส่วนนี้: แปลงคอลัมน์วันที่ให้คอมพิวเตอร์รู้จักว่าเป็นวันที่ ตรวจค่าว่างหลังแก้ไข และจัดทำตารางบันทึกผลก่อนและหลังการแก้ไข
- เหตุผลที่ต้องแปลงวันที่: ขณะนี้คอมพิวเตอร์ถือว่า 2026-06-01 เป็นเพียงข้อความ จึงเรียงลำดับหรือคำนวณระยะเวลาไม่ได้ เมื่อแปลงเป็นชนิดวันที่ (datetime) แล้วจึงนำไปคำนวณได้
- ข้อควรทราบเรื่องวันที่: การแปลงได้ไม่ได้หมายความว่าข้อมูลเป็นปัจจุบัน ชุดข้อมูลนี้มี 2 แถวที่ปรับปรุงล่าสุดเมื่อปี 2020 (S031 และ S034) ซึ่งควรตรวจสอบเพิ่มเติม
- บันทึกการแก้ไขข้อมูล: เหมือนสมุดบันทึกการซ่อมรถที่ระบุว่าก่อนซ่อมและหลังซ่อมเป็นอย่างไร เพื่อให้ผู้อื่นตรวจสอบย้อนหลังได้ว่าแก้ไขอะไรไปบ้าง


## ขั้นตอนที่ 26: แปลงคอลัมน์วันที่ให้เป็นชนิดวันที่

แปลงคอลัมน์ updated_date จากข้อความให้เป็นชนิดวันที่ (datetime) เพื่อให้เรียงลำดับและคำนวณระยะเวลาได้ แล้วตรวจสอบชนิดข้อมูลของทุกคอลัมน์

```python
clean['updated_date'] = pd.to_datetime(clean['updated_date'], format='%Y-%m-%d', errors='coerce')
clean.dtypes
```

- `pd.to_datetime(...)`: แปลงข้อความให้เป็นวันที่
- `format='%Y-%m-%d'`: บอกรูปแบบของข้อความ คือ ปี ค.ศ. 4 หลัก-เดือน-วัน เช่น 2026-06-01
- `errors='coerce'`: หากพบข้อความที่แปลงเป็นวันที่ไม่ได้ ให้เปลี่ยนเป็นค่าว่าง (NaT) แทนการหยุดทำงาน
- `clean['updated_date'] =`: บันทึกผลกลับไปที่คอลัมน์เดิม
- `clean.dtypes`: แสดงชนิดข้อมูลของทุกคอลัมน์ เพื่อตรวจว่าแปลงสำเร็จ

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [26]:
clean['updated_date'] = pd.to_datetime(clean['updated_date'], format='%Y-%m-%d', errors='coerce')
clean.dtypes

student_id              object
name                    object
major                   object
age                    float64
gpa                    float64
phone           string[python]
updated_date    datetime64[ns]
dtype: object

### ผลลัพธ์และคำอธิบาย

```text
student_id              object
name                    object
major                   object
age                    float64
gpa                    float64
phone           string[python]
updated_date    datetime64[ns]
```

คอลัมน์ updated_date แสดงชนิดข้อมูลเป็น datetime64[ns] หมายความว่าแปลงเป็นวันที่สำเร็จ ส่วนคอลัมน์อื่นยังเป็นชนิดเดิม

**ข้อควรระวังในการเขียนคำสั่ง:** ตัวอักษร Y ต้องเป็นตัวพิมพ์ใหญ่ ส่วน m และ d ต้องเป็นตัวพิมพ์เล็ก หากพิมพ์ผิด วันที่จะกลายเป็นค่าว่างทั้งคอลัมน์

**คำถามตรวจสอบความเข้าใจ:** ตรวจสอบได้อย่างไรว่าคอลัมน์ updated_date แปลงเป็นวันที่สำเร็จแล้ว


## ขั้นตอนที่ 27: ตรวจจำนวนค่าว่างหลังการแก้ไข

นับจำนวนช่องที่ว่างในแต่ละคอลัมน์ของตาราง clean เพื่อตรวจผลการแก้ไขทั้งหมด

```python
clean.isna().sum()
```

- `clean.isna()`: ตรวจทุกช่องว่าเป็นค่าว่างหรือไม่
- `.sum()`: นับจำนวนช่องที่ว่างแยกตามคอลัมน์

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [27]:
clean.isna().sum()

student_id      0
name            0
major           0
age             2
gpa             3
phone           0
updated_date    0
dtype: int64

### ผลลัพธ์และคำอธิบาย

```text
student_id      0
name            0
major           0
age             2
gpa             3
phone           0
updated_date    0
```

age ว่าง 2 แถว (S011 เดิม และ S008 ที่พักค่าไว้) gpa ว่าง 3 แถว (S004 และ S023 เดิม และ S019 ที่พักค่าไว้) phone ว่าง 0 และ updated_date ว่าง 0

**ข้อควรระวังในการเขียนคำสั่ง:** phone ว่าง 0 ไม่ได้หมายความว่ามีหมายเลขโทรศัพท์ครบทุกคน เพราะ S016 ถูกระบุว่า ไม่ระบุ ในวันจันทร์

**คำถามตรวจสอบความเข้าใจ:** เหตุใดอายุและเกรดเฉลี่ยจึงมีค่าว่างมากกว่าก่อนแก้ไข


## ขั้นตอนที่ 28: จัดทำตารางบันทึกการแก้ไขข้อมูล

สร้างตารางเปรียบเทียบตัวเลขก่อนการแก้ไข (จากตาราง df) และหลังการแก้ไข (จากตาราง clean) เพื่อเป็นหลักฐานว่าแก้ไขอะไรไปบ้าง

```python
cleaning_log = pd.DataFrame({
    'item': ['จำนวนแถว', 'อายุที่ว่าง', 'เกรดเฉลี่ยที่ว่าง', 'หมายเลขโทรศัพท์ที่ว่าง'],
    'before': [len(df), df['age'].isna().sum(), df['gpa'].isna().sum(), df['phone'].isna().sum()],
    'after': [len(clean), clean['age'].isna().sum(), clean['gpa'].isna().sum(), clean['phone'].isna().sum()]
})
cleaning_log
```

- `'item': [...]`: คอลัมน์รายการที่ตรวจ มี 4 รายการ
- `'before': [...]`: ตัวเลขก่อนการแก้ไข นับจากตาราง df
- `'after': [...]`: ตัวเลขหลังการแก้ไข นับจากตาราง clean
- `cleaning_log =`: ตั้งชื่อตารางบันทึกการแก้ไขว่า cleaning_log

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [28]:
cleaning_log = pd.DataFrame({
    'item': ['จำนวนแถว', 'อายุที่ว่าง', 'เกรดเฉลี่ยที่ว่าง', 'หมายเลขโทรศัพท์ที่ว่าง'],
    'before': [len(df), df['age'].isna().sum(), df['gpa'].isna().sum(), df['phone'].isna().sum()],
    'after': [len(clean), clean['age'].isna().sum(), clean['gpa'].isna().sum(), clean['phone'].isna().sum()]
})
cleaning_log

,item,before,after
0,จำนวนแถว,42,40
1,อายุที่ว่าง,1,2
2,เกรดเฉลี่ยที่ว่าง,2,3
3,หมายเลขโทรศัพท์ที่ว่าง,1,0


### ผลลัพธ์และคำอธิบาย

| ลำดับแถว | item | before | after |
| --- | --- | --- | --- |
| 0 | จำนวนแถว | 42 | 40 |
| 1 | อายุที่ว่าง | 1 | 2 |
| 2 | เกรดเฉลี่ยที่ว่าง | 2 | 3 |
| 3 | หมายเลขโทรศัพท์ที่ว่าง | 1 | 0 |

จำนวนแถว 42 → 40 อายุที่ว่าง 1 → 2 เกรดเฉลี่ยที่ว่าง 2 → 3 และหมายเลขโทรศัพท์ที่ว่าง 1 → 0

**ข้อควรระวังในการเขียนคำสั่ง:** ลำดับรายการในทุกคอลัมน์ต้องตรงกัน คอลัมน์ before ต้องนับจาก df และคอลัมน์ after ต้องนับจาก clean หากสลับกัน ตัวเลขจะกลับด้าน

**คำถามตรวจสอบความเข้าใจ:** อธิบายเหตุผลที่ตัวเลขแต่ละแถวเปลี่ยนไปได้ เช่น จำนวนแถวลดลง 2 แถวเพราะลบแถวซ้ำ


## ส่วนที่ 4: ส่งออกไฟล์สำหรับเผยแพร่ (ขั้นตอนที่ 29–31)

- เป้าหมายของส่วนนี้: ส่งออกไฟล์ที่แบ่งปันให้ผู้อื่นได้อย่างปลอดภัย โดยไม่มีข้อมูลส่วนบุคคล
- เหตุผลที่ไม่ส่งตาราง clean: ตาราง clean ยังมีชื่อ รหัสผู้เรียน และหมายเลขโทรศัพท์ ซึ่งเป็นข้อมูลส่วนบุคคลที่ต้องคุ้มครองตามพระราชบัญญัติคุ้มครองข้อมูลส่วนบุคคล (PDPA)
- สิ่งที่เผยแพร่: เฉพาะตารางบันทึกการแก้ไข ซึ่งมีแต่จำนวนรวมก่อนและหลังการแก้ไข ไม่ระบุตัวบุคคล
- ข้อควรทราบ: การปิดบังเพียงบางส่วน เช่น ซ่อนหมายเลขโทรศัพท์ อาจยังระบุตัวบุคคลได้จากข้อมูลอื่น จึงควรเผยแพร่เฉพาะข้อมูลที่จำเป็นและตรวจไฟล์จริงทุกครั้งก่อนแบ่งปัน ทั้งนี้ขั้นตอนในบทเรียนไม่ได้รับรองว่าปฏิบัติตามกฎหมายครบถ้วน


## ขั้นตอนที่ 29: เตรียมตารางสำหรับเผยแพร่

สร้างตาราง public จากตารางบันทึกการแก้ไข ตารางนี้มีเฉพาะจำนวนรวม จึงเผยแพร่ได้โดยไม่เปิดเผยข้อมูลส่วนบุคคล

```python
public = cleaning_log.copy()
public
```

- `cleaning_log.copy()`: สร้างสำเนาของตารางบันทึกการแก้ไข
- `public =`: ตั้งชื่อตารางที่จะเผยแพร่ว่า public
- `public`: บรรทัดสุดท้ายแสดงตารางเพื่อตรวจเนื้อหา

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [29]:
public = cleaning_log.copy()
public

,item,before,after
0,จำนวนแถว,42,40
1,อายุที่ว่าง,1,2
2,เกรดเฉลี่ยที่ว่าง,2,3
3,หมายเลขโทรศัพท์ที่ว่าง,1,0


### ผลลัพธ์และคำอธิบาย

| ลำดับแถว | item | before | after |
| --- | --- | --- | --- |
| 0 | จำนวนแถว | 42 | 40 |
| 1 | อายุที่ว่าง | 1 | 2 |
| 2 | เกรดเฉลี่ยที่ว่าง | 2 | 3 |
| 3 | หมายเลขโทรศัพท์ที่ว่าง | 1 | 0 |

ตารางมี 4 แถว 3 คอลัมน์ (item, before และ after) ไม่มีชื่อ รหัสผู้เรียน หรือหมายเลขโทรศัพท์

**ข้อควรระวังในการเขียนคำสั่ง:** ห้ามใช้ตาราง clean เป็นไฟล์เผยแพร่ เพราะยังมีข้อมูลส่วนบุคคลของผู้เรียนทุกคน

**คำถามตรวจสอบความเข้าใจ:** ตาราง public มีข้อมูลที่ระบุตัวบุคคลได้หรือไม่


## ขั้นตอนที่ 30: บันทึกไฟล์สำหรับเผยแพร่

บันทึกตาราง public เป็นไฟล์ชื่อ clean_public.csv เพื่อนำไปแบ่งปันได้

```python
public.to_csv('clean_public.csv', index=False, encoding='utf-8-sig')
```

- `public.to_csv('clean_public.csv', ...)`: บันทึกตาราง public เป็นไฟล์ CSV ตามชื่อที่กำหนด
- `index=False`: ไม่บันทึกเลขลำดับแถวลงในไฟล์
- `encoding='utf-8-sig'`: กำหนดรูปแบบการบันทึกให้ภาษาไทยแสดงผลถูกต้องในโปรแกรม Excel

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [30]:
public.to_csv('clean_public.csv', index=False, encoding='utf-8-sig')

### ผลลัพธ์และคำอธิบาย

```text
รันคำสั่งสำเร็จ (ไม่มีข้อความแจ้งข้อผิดพลาด)
```

ขั้นตอนนี้ไม่แสดงตาราง ให้คลิกรูปโฟลเดอร์ทางด้านซ้าย จะพบไฟล์ clean_public.csv (หากไม่พบ ให้คลิกปุ่มรีเฟรชในแถบ Files)

**ข้อควรระวังในการเขียนคำสั่ง:** ไฟล์ที่สร้างใน Colab จะถูกลบเมื่อปิดหน้าต่าง หากต้องการเก็บไว้ ให้คลิกขวาที่ไฟล์แล้วเลือก Download

**คำถามตรวจสอบความเข้าใจ:** พบไฟล์ clean_public.csv ในแถบ Files ทางด้านซ้าย


## ขั้นตอนที่ 31: อ่านไฟล์ที่บันทึกกลับมาตรวจสอบ

อ่านไฟล์ clean_public.csv ที่เพิ่งบันทึกกลับเข้ามา เพื่อตรวจเนื้อหาในไฟล์จริงก่อนแบ่งปันให้ผู้อื่น

```python
public_saved = pd.read_csv('clean_public.csv')
public_saved
```

- `pd.read_csv('clean_public.csv')`: อ่านไฟล์ที่เพิ่งบันทึก
- `public_saved =`: ตั้งชื่อตารางที่อ่านกลับมาว่า public_saved
- `public_saved`: บรรทัดสุดท้ายแสดงตารางเพื่อตรวจเนื้อหา

พิมพ์คำสั่งและกดปุ่ม Shift + Enter เพื่อรันคำสั่งและตรวจสอบผลลัพธ์


In [31]:
public_saved = pd.read_csv('clean_public.csv')
public_saved

,item,before,after
0,จำนวนแถว,42,40
1,อายุที่ว่าง,1,2
2,เกรดเฉลี่ยที่ว่าง,2,3
3,หมายเลขโทรศัพท์ที่ว่าง,1,0


### ผลลัพธ์และคำอธิบาย

| ลำดับแถว | item | before | after |
| --- | --- | --- | --- |
| 0 | จำนวนแถว | 42 | 40 |
| 1 | อายุที่ว่าง | 1 | 2 |
| 2 | เกรดเฉลี่ยที่ว่าง | 2 | 3 |
| 3 | หมายเลขโทรศัพท์ที่ว่าง | 1 | 0 |

แสดงตาราง 4 แถว 3 คอลัมน์ เหมือนตาราง public ทุกประการ และไม่มีคอลัมน์ที่เป็นข้อมูลส่วนบุคคล

**ข้อควรระวังในการเขียนคำสั่ง:** ต้องตรวจเนื้อหาในไฟล์จริงทุกครั้งก่อนแบ่งปัน ไม่ควรตรวจเพียงชื่อไฟล์หรือชื่อตาราง

**คำถามตรวจสอบความเข้าใจ:** ไฟล์ใดแบ่งปันให้ผู้อื่นได้ และตารางใดต้องเก็บไว้ใน Notebook ส่วนตัวเท่านั้น


## สรุป

อธิบายคำสั่ง ผลลัพธ์ หน่วย และข้อจำกัด บันทึก Notebook สำหรับทบทวน
